In [1]:
import os

# 必须在 import sentence_transformers 之前设置！
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

print("HF_ENDPOINT =", os.environ["HF_ENDPOINT"])
print("CA 证书 =", certifi.where())


HF_ENDPOINT = https://hf-mirror.com
CA 证书 = e:\111\Lib\site-packages\certifi\cacert.pem


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
key=os.getenv("DEEPSEEK_API_KEY")

In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "用一句话解释什么是 RAG，不超过 50 字。"}
    ],
)

print(resp.choices[0].message.content)

RAG是检索增强生成：先从外部知识库检索相关信息，再交给大模型生成回答。


In [4]:
from pathlib import Path

data_dir=Path('data')
docs=[]
for path in sorted(data_dir.glob('*.md')):
    text=path.read_text(encoding='utf-8')
    docs.append((path.name,text))
print(f"Loaded {len(docs)} documents from {data_dir.resolve()}")


Loaded 5 documents from E:\rag_learn\data


In [5]:
def spilit_text(text,chunk_size=300,overlap=50):
    chunks=[]
    step=chunk_size-overlap
    start=0
    while start<len(text):
        piece=text[start:start+chunk_size].strip()
        if piece:
            chunks.append(piece)
        start+=step
    return chunks

all_chunks=[]
for name,text in docs:
    pieces=spilit_text(text) 
    for i ,piece in enumerate(pieces):
        all_chunks.append(
            {
                "doc":name,
                "chunk_id":i,
                "text":piece
            }
        )
print(f"Loaded {len(all_chunks)} chunks from {data_dir.resolve()}")


Loaded 9 chunks from E:\rag_learn\data


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

texts = [c["text"] for c in all_chunks]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,   # 归一化，之后点积就等于余弦相似度
    show_progress_bar=True,
)

print("chunk 数量:", len(texts))
print("embeddings 形状:", embeddings.shape)
print("第一条向量的前 5 个数字:", embeddings[0][:5])


e:\111\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\111\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\AI_Cache\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-

chunk 数量: 9
embeddings 形状: (9, 512)
第一条向量的前 5 个数字: [ 0.01999746  0.04311125  0.06498501  0.02530182 -0.03316524]


In [9]:
import numpy as np

def retrieve(query, top_k=3):
    """输入问题，返回最相关的 top_k 个 chunk（含分数）。"""
    q_vec = model.encode(query, normalize_embeddings=True)
    scores = embeddings @ q_vec              # (9, 512) @ (512,) -> (9,)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "score": float(scores[idx]),
            "doc": all_chunks[idx]["doc"],
            "chunk_id": all_chunks[idx]["chunk_id"],
            "text": all_chunks[idx]["text"],
        })
    return results


# 试一个问题
query = "X1 和 X2 哪个续航更长？"
results = retrieve(query, top_k=3)

for rank, r in enumerate(results, start=1):
    print(f"\n第 {rank} 名 | 相似度 {r['score']:.4f} | {r['doc']} #{r['chunk_id']}")
    print(r["text"][:150].replace("\n", " "))



第 1 名 | 相似度 0.5805 | 05_常见问题FAQ.md #0
# 常见问题 FAQ  ## 手表可以连接蓝牙耳机吗？  X2 支持连接蓝牙耳机并播放离线音乐。X1 不支持连接蓝牙耳机。  ## 续航时间为什么和宣传不一致？  实际续航受屏幕亮度、运动记录频率、消息通知数量等因素影响。建议在设置中开启「智能续航模式」。  ## 如何同步数据？  下载「星河健康」

第 2 名 | 相似度 0.5399 | 02_智能手表X1产品手册.md #0
# 星河智能手表 X1 产品手册  ## 基本参数  - 屏幕：1.43 英寸 AMOLED，分辨率 466×466 - 重量：38 克（不含表带） - 续航：典型使用 14 天，重度使用 7 天 - 防水：5 ATM，可佩戴游泳，但不支持潜水 - 传感器：心率、血氧、加速度计、陀螺仪  ## 主要

第 3 名 | 相似度 0.5255 | 03_智能手表X2产品手册.md #0
# 星河智能手表 X2 产品手册  ## 基本参数  - 屏幕：1.5 英寸 AMOLED，分辨率 480×480 - 重量：45 克（不含表带） - 续航：典型使用 10 天，重度使用 5 天 - 防水：5 ATM，可佩戴游泳 - 传感器：心率、血氧、气压计、GPS、加速度计、陀螺仪  ## 主要


In [10]:
def build_prompt(query, results):
    """把检索到的 chunk 拼成带编号的资料块。"""
    context = ""
    for i, r in enumerate(results, start=1):
        context += f"[{i}] 来源：{r['doc']} #{r['chunk_id']}\n{r['text']}\n\n"

    system_prompt = (
        "你是一个严谨的问答助手。请只根据用户提供的资料回答问题。"
        "如果资料中没有答案，请明确回答「根据现有资料无法回答」，不要编造。"
        "回答时请用 [1]、[2] 这样的编号标注你依据的资料。"
    )

    user_prompt = f"资料：\n{context}问题：{query}"

    return system_prompt, user_prompt


In [11]:
def rag_answer(query, top_k=3):
    # 1. 检索
    results = retrieve(query, top_k=top_k)

    # 2. 拼 Prompt
    system_prompt, user_prompt = build_prompt(query, results)

    # 3. 调模型
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,      # 0 表示尽量确定，适合问答
    )

    answer = resp.choices[0].message.content
    return answer, results


In [12]:
answer, results = rag_answer("X1 手表的续航时间是多久？")

print("回答：\n", answer)

print("\n参考来源：")
for i, r in enumerate(results, start=1):
    print(f"[{i}] {r['doc']} #{r['chunk_id']}  相似度 {r['score']:.3f}")


回答：
 根据资料，X1 手表的续航时间为：典型使用 14 天，重度使用 7 天。[1]

参考来源：
[1] 02_智能手表X1产品手册.md #0  相似度 0.688
[2] 03_智能手表X2产品手册.md #0  相似度 0.660
[3] 05_常见问题FAQ.md #0  相似度 0.632


In [13]:
answer, _ = rag_answer("星河手表的表带是什么材质的？")
print(answer)


根据现有资料无法回答。资料中只提到了表带的价格（X2 表带 129 元 [1]，X1 表带 99 元 [2]）以及表带属于易耗品、不在保修范围内 [3]，但没有说明表带的材质。


In [15]:
answer, _ = rag_answer("X1 和 X2 哪个续航更长？", top_k=5)
print(answer)


根据现有资料，X1 的续航更长。

X1 典型使用续航为 14 天、重度使用为 7 天[2]；X2 典型使用续航为 10 天、重度使用为 5 天[3]。


In [16]:
resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "X1 手表的续航时间是多久？"}],
    temperature=0,
)
print(resp.choices[0].message.content)


您提到的“X1 手表”可能指代多个品牌的产品（如荣耀手表 X1、某些智能穿戴品牌的 X1 系列等），不同产品的续航时间差异较大。为了给您准确的答案，需要明确具体品牌和型号。

以下是几种常见“X1 手表”的续航参考：

1. **荣耀手表 X1**：典型使用场景下续航约 **7-10 天**（具体取决于使用频率和功能开启情况）。
2. **部分儿童/老人定位手表 X1**：续航通常在 **2-5 天**，视定位频率和通话使用而定。
3. **其他品牌 X1 系列智能手表**：有的主打长续航可达 **14 天以上**，有的全能模式约 **3-5 天**。

如果您能提供手表的**品牌全称**或**产品链接/图片**，我可以帮您查到更精确的官方续航数据。
